In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
import logging
import sqlite3
import pandas as pd


@dataclass
class PipelineConfig:
    input_path: str
    output_database: str = "retail_dw.db"
    batch_list: list[int] = field(default_factory=lambda: [1, 2, 3])
    error_mode: str = "continue"   # continue | fail


PAYMENT_MAP = {
    "cash": "Cash",
    "promptpay": "PromptPay",
    "credit card": "Credit Card",
    "bank transfer": "Bank Transfer",
}

CHANNEL_MAP = {
    "online": "Online",
    "e-commerce": "Online",
    "marketplace": "Marketplace",
    "store": "Store",
}


def setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )


def read_dimension_data(config: PipelineConfig):
    xlsx = Path(config.input_path)
    customers = pd.read_excel(xlsx, sheet_name="customers", dtype=str)
    products = pd.read_excel(xlsx, sheet_name="products", dtype=str)

    customers["customer_id"] = customers["customer_id"].astype("string").str.strip()
    products["product_id"] = products["product_id"].astype("string").str.strip()

    return customers, products


def extract_batch(config: PipelineConfig, batch_no: int) -> pd.DataFrame:
    started = datetime.now()
    sheet = f"orders_batch_{batch_no}"
    try:
        df = pd.read_excel(config.input_path, sheet_name=sheet)
        df["source_batch"] = batch_no
        logging.info(
            "EXTRACT %s | rows=%d | started=%s | ended=%s",
            sheet, len(df), started.isoformat(timespec="seconds"),
            datetime.now().isoformat(timespec="seconds"),
        )
        return df
    except Exception:
        logging.exception("EXTRACT FAILED %s", sheet)
        if config.error_mode == "fail":
            raise
        return pd.DataFrame()


def normalize_payment(value):
    if pd.isna(value):
        return None
    key = str(value).strip().lower()
    return PAYMENT_MAP.get(key)


def normalize_channel(value):
    if pd.isna(value):
        return None
    key = str(value).strip().lower()
    return CHANNEL_MAP.get(key)


def parse_price(value):
    if pd.isna(value):
        return pd.NA
    s = str(value).strip().replace(",", "")
    # Dataset intentionally contains values such as "THB 979.40".
    # Strip the currency label, then parse safely.
    s = s.replace("THB", "").strip()
    return pd.to_numeric(s, errors="coerce")


def transform_validate(raw, customers, products):
    df = raw.copy()

    # Keep original values for quarantine reasons.
    df["_row_number"] = range(1, len(df) + 1)
    reasons = [[] for _ in range(len(df))]

    def add_reason(mask, code):
        for i in df.index[mask.fillna(False)]:
            reasons[i].append(code)

    # Safe type conversion.
    df["order_datetime"] = pd.to_datetime(df["order_datetime"], errors="coerce")
    df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce")
    df["quantity_num"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price_num"] = df["unit_price"].map(parse_price)
    df["discount_num"] = pd.to_numeric(df["discount_pct"], errors="coerce")

    add_reason(df["order_id"].isna() | df["order_id"].astype("string").str.strip().eq(""), "MISSING_ORDER_ID")
    add_reason(df["order_datetime"].isna(), "INVALID_ORDER_DATETIME")
    add_reason(df["updated_at"].isna(), "INVALID_UPDATED_AT")
    add_reason(df["customer_id"].isna() | df["customer_id"].astype("string").str.strip().eq(""), "MISSING_CUSTOMER_ID")
    add_reason(df["product_id"].isna() | df["product_id"].astype("string").str.strip().eq(""), "MISSING_PRODUCT_ID")
    add_reason(df["quantity_num"].isna(), "INVALID_QUANTITY_TYPE")
    add_reason(df["quantity_num"].notna() & (df["quantity_num"] <= 0), "INVALID_QUANTITY")
    add_reason(df["quantity_num"].notna() & (df["quantity_num"] != df["quantity_num"].round()), "INVALID_QUANTITY_TYPE")
    add_reason(df["unit_price_num"].isna(), "INVALID_UNIT_PRICE")
    add_reason(df["unit_price_num"].notna() & (df["unit_price_num"] <= 0), "INVALID_UNIT_PRICE")
    add_reason(df["discount_num"].isna(), "INVALID_DISCOUNT_TYPE")
    add_reason(df["discount_num"].notna() & ~df["discount_num"].between(0, 100), "INVALID_DISCOUNT")
    add_reason(df["payment_method"].map(normalize_payment).isna(), "INVALID_PAYMENT_METHOD")
    add_reason(df["sales_channel"].map(normalize_channel).isna(), "INVALID_SALES_CHANNEL")

    customer_ids = set(customers["customer_id"].dropna())
    product_ids = set(products["product_id"].dropna())

    add_reason(
        df["customer_id"].notna() & ~df["customer_id"].astype("string").isin(customer_ids),
        "UNKNOWN_CUSTOMER_ID",
    )
    add_reason(
        df["product_id"].notna() & ~df["product_id"].astype("string").isin(product_ids),
        "UNKNOWN_PRODUCT_ID",
    )

    df["reason_code"] = [
        "|".join(dict.fromkeys(r)) if r else ""
        for r in reasons
    ]

    quarantine = df[df["reason_code"] != ""].copy()

    clean = df[df["reason_code"] == ""].copy()
    clean["quantity"] = clean["quantity_num"].astype(int)
    clean["unit_price"] = clean["unit_price_num"].astype(float)
    clean["discount_pct"] = clean["discount_num"].astype(float)
    clean["payment_method"] = clean["payment_method"].map(normalize_payment)
    clean["sales_channel"] = clean["sales_channel"].map(normalize_channel)

    clean["gross_amount"] = clean["quantity"] * clean["unit_price"]
    clean["net_amount"] = clean["gross_amount"] * (1 - clean["discount_pct"] / 100)

    clean["order_id"] = clean["order_id"].astype(str).str.strip()
    clean["customer_id"] = clean["customer_id"].astype(str).str.strip()
    clean["product_id"] = clean["product_id"].astype(str).str.strip()

    # Deduplicate by order_id, keeping latest updated_at.
    clean = clean.sort_values(["order_id", "updated_at", "_row_number"])
    duplicate_mask = clean.duplicated("order_id", keep="last")
    duplicated_count = int(duplicate_mask.sum())
    clean = clean.loc[~duplicate_mask].copy()

    return clean, quarantine, duplicated_count


def create_database(db_path: str):
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")

    conn.executescript("""
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id TEXT NOT NULL UNIQUE,
        customer_name TEXT,
        province TEXT,
        segment TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_product (
        product_key INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id TEXT NOT NULL UNIQUE,
        product_name TEXT,
        category TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_date (
        date_key INTEGER PRIMARY KEY,
        full_date TEXT NOT NULL UNIQUE,
        day INTEGER,
        month INTEGER,
        quarter INTEGER,
        year INTEGER
    );

    CREATE TABLE IF NOT EXISTS fact_sales (
        order_id TEXT PRIMARY KEY,
        date_key INTEGER NOT NULL,
        customer_key INTEGER NOT NULL,
        product_key INTEGER NOT NULL,
        quantity INTEGER NOT NULL CHECK(quantity > 0),
        unit_price REAL NOT NULL CHECK(unit_price > 0),
        discount_pct REAL NOT NULL CHECK(discount_pct BETWEEN 0 AND 100),
        gross_amount REAL NOT NULL,
        net_amount REAL NOT NULL CHECK(net_amount >= 0),
        payment_method TEXT NOT NULL,
        sales_channel TEXT NOT NULL,
        updated_at TEXT NOT NULL,
        source_batch INTEGER NOT NULL,
        FOREIGN KEY(date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY(customer_key) REFERENCES dim_customer(customer_key),
        FOREIGN KEY(product_key) REFERENCES dim_product(product_key)
    );

    CREATE TABLE IF NOT EXISTS quarantine (
        quarantine_id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_batch INTEGER NOT NULL,
        row_number INTEGER,
        order_id TEXT,
        reason_code TEXT NOT NULL,
        raw_data TEXT NOT NULL,
        rejected_at TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS pipeline_run_log (
        run_id INTEGER PRIMARY KEY AUTOINCREMENT,
        batch INTEGER NOT NULL,
        started_at TEXT NOT NULL,
        ended_at TEXT NOT NULL,
        rows_read INTEGER NOT NULL,
        rows_valid INTEGER NOT NULL,
        rows_rejected INTEGER NOT NULL,
        rows_duplicated INTEGER NOT NULL,
        rows_loaded INTEGER NOT NULL,
        total_net_sales REAL NOT NULL,
        status TEXT NOT NULL
    );
    """)
    conn.commit()
    return conn


def load_dimensions(conn, customers, products):
    for _, r in customers.iterrows():
        conn.execute("""
            INSERT OR IGNORE INTO dim_customer
            (customer_id, customer_name, province, segment)
            VALUES (?, ?, ?, ?)
        """, (
            r["customer_id"], r["customer_name"], r["province"], r["segment"]
        ))

    for _, r in products.iterrows():
        conn.execute("""
            INSERT OR IGNORE INTO dim_product
            (product_id, product_name, category)
            VALUES (?, ?, ?)
        """, (
            r["product_id"], r["product_name"], r["category"]
        ))
    conn.commit()


def get_dimension_keys(conn):
    customers = {
        r[0]: r[1]
        for r in conn.execute("SELECT customer_id, customer_key FROM dim_customer")
    }
    products = {
        r[0]: r[1]
        for r in conn.execute("SELECT product_id, product_key FROM dim_product")
    }
    return customers, products


def upsert_date(conn, date_value):
    full_date = pd.Timestamp(date_value).strftime("%Y-%m-%d")
    d = pd.Timestamp(date_value)
    date_key = int(d.strftime("%Y%m%d"))
    conn.execute("""
        INSERT OR IGNORE INTO dim_date
        (date_key, full_date, day, month, quarter, year)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (date_key, full_date, d.day, d.month, ((d.month - 1)//3) + 1, d.year))
    return date_key


def load_fact(conn, clean):
    customer_keys, product_keys = get_dimension_keys(conn)
    loaded = 0

    for _, r in clean.iterrows():
        # Incremental + idempotent behavior:
        # replace only when incoming updated_at is newer.
        existing = conn.execute(
            "SELECT updated_at FROM fact_sales WHERE order_id = ?",
            (r["order_id"],)
        ).fetchone()

        incoming_updated = pd.Timestamp(r["updated_at"]).isoformat(sep=" ")
        if existing is not None and incoming_updated <= existing[0]:
            continue

        date_key = upsert_date(conn, r["order_datetime"])
        customer_key = customer_keys[r["customer_id"]]
        product_key = product_keys[r["product_id"]]

        conn.execute("""
            INSERT INTO fact_sales (
                order_id, date_key, customer_key, product_key,
                quantity, unit_price, discount_pct,
                gross_amount, net_amount,
                payment_method, sales_channel,
                updated_at, source_batch
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(order_id) DO UPDATE SET
                date_key=excluded.date_key,
                customer_key=excluded.customer_key,
                product_key=excluded.product_key,
                quantity=excluded.quantity,
                unit_price=excluded.unit_price,
                discount_pct=excluded.discount_pct,
                gross_amount=excluded.gross_amount,
                net_amount=excluded.net_amount,
                payment_method=excluded.payment_method,
                sales_channel=excluded.sales_channel,
                updated_at=excluded.updated_at,
                source_batch=excluded.source_batch
        """, (
            r["order_id"], date_key, customer_key, product_key,
            int(r["quantity"]), float(r["unit_price"]), float(r["discount_pct"]),
            float(r["gross_amount"]), float(r["net_amount"]),
            r["payment_method"], r["sales_channel"],
            incoming_updated, int(r["source_batch"])
        ))
        loaded += 1

    return loaded


def write_quarantine(conn, quarantine):
    if quarantine.empty:
        return

    now = datetime.now().isoformat(timespec="seconds")
    for _, r in quarantine.iterrows():
        raw = {
            k: (None if pd.isna(v) else str(v))
            for k, v in r.items()
            if k not in {"reason_code"}
        }
        conn.execute("""
            INSERT INTO quarantine
            (source_batch, row_number, order_id, reason_code, raw_data, rejected_at)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            int(r["source_batch"]),
            int(r["_row_number"]),
            None if pd.isna(r["order_id"]) else str(r["order_id"]),
            r["reason_code"],
            str(raw),
            now
        ))


def run_pipeline(config: PipelineConfig):
    setup_logging()

    if config.error_mode not in {"continue", "fail"}:
        raise ValueError("error_mode must be 'continue' or 'fail'")

    customers, products = read_dimension_data(config)
    conn = create_database(config.output_database)
    load_dimensions(conn, customers, products)

    for batch_no in config.batch_list:
        started = datetime.now()

        try:
            raw = extract_batch(config, batch_no)
            if raw.empty:
                raise RuntimeError("Batch is empty or could not be read")

            clean, quarantine, duplicated_count = transform_validate(
                raw, customers, products
            )

            rows_read = len(raw)
            rows_rejected = len(quarantine)
            rows_valid = len(clean) + duplicated_count

            write_quarantine(conn, quarantine)
            loaded = load_fact(conn, clean)

            total_net = float(
                conn.execute(
                    "SELECT COALESCE(SUM(net_amount), 0) FROM fact_sales"
                ).fetchone()[0]
            )

            conn.commit()

            ended = datetime.now()
            conn.execute("""
                INSERT INTO pipeline_run_log
                (batch, started_at, ended_at, rows_read, rows_valid,
                 rows_rejected, rows_duplicated, rows_loaded,
                 total_net_sales, status)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                batch_no,
                started.isoformat(timespec="seconds"),
                ended.isoformat(timespec="seconds"),
                rows_read,
                rows_valid,
                rows_rejected,
                duplicated_count,
                loaded,
                total_net,
                "SUCCESS"
            ))
            conn.commit()

            logging.info(
                "BATCH %d | read=%d valid=%d rejected=%d duplicated=%d loaded=%d net_sales=%.2f",
                batch_no, rows_read, rows_valid, rows_rejected,
                duplicated_count, loaded, total_net
            )

        except Exception as exc:
            conn.rollback()
            ended = datetime.now()

            conn.execute("""
                INSERT INTO pipeline_run_log
                (batch, started_at, ended_at, rows_read, rows_valid,
                 rows_rejected, rows_duplicated, rows_loaded,
                 total_net_sales, status)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                batch_no,
                started.isoformat(timespec="seconds"),
                ended.isoformat(timespec="seconds"),
                0, 0, 0, 0, 0, 0.0,
                f"FAILED: {type(exc).__name__}"
            ))
            conn.commit()

            logging.exception("BATCH %d FAILED", batch_no)
            if config.error_mode == "fail":
                conn.close()
                raise

    conn.close()


def print_final_report(db_path):
    conn = sqlite3.connect(db_path)

    print("\n=== PIPELINE RUN LOG ===")
    print(pd.read_sql_query("SELECT * FROM pipeline_run_log ORDER BY run_id", conn).to_string(index=False))

    print("\n=== FINAL KPI ===")
    kpi = pd.read_sql_query("""
        SELECT
            (SELECT COUNT(*) FROM fact_sales) AS fact_rows,
            (SELECT COUNT(*) FROM quarantine) AS rejected_rows,
            (SELECT COALESCE(SUM(net_amount),0) FROM fact_sales) AS total_net_sales,
            (SELECT COUNT(*) FROM fact_sales
             WHERE quantity <= 0 OR unit_price <= 0 OR net_amount < 0) AS invalid_fact_rows
    """, conn)
    print(kpi.to_string(index=False))

    conn.close()


if __name__ == "__main__":
    config = PipelineConfig(
        input_path="Python_Data_Pipeline_Lab_Dataset.xlsx",
        output_database="retail_dw.db",
        batch_list=[1, 1, 2, 3],  # batch_1, batch_1 repeat, batch_2, batch_3
        error_mode="continue",
    )
    run_pipeline(config)
    print_final_report(config.output_database)

2026-08-18 14:56:31,158 | INFO | EXTRACT orders_batch_1 | rows=420 | started=2026-08-18T14:56:30 | ended=2026-08-18T14:56:31
2026-08-18 14:56:31,336 | INFO | BATCH 1 | read=420 valid=389 rejected=31 duplicated=0 loaded=389 net_sales=982277.68
2026-08-18 14:56:31,686 | INFO | EXTRACT orders_batch_1 | rows=420 | started=2026-08-18T14:56:31 | ended=2026-08-18T14:56:31
2026-08-18 14:56:31,816 | INFO | BATCH 1 | read=420 valid=389 rejected=31 duplicated=0 loaded=0 net_sales=982277.68
2026-08-18 14:56:32,096 | INFO | EXTRACT orders_batch_2 | rows=424 | started=2026-08-18T14:56:31 | ended=2026-08-18T14:56:32
2026-08-18 14:56:32,266 | INFO | BATCH 2 | read=424 valid=387 rejected=37 duplicated=1 loaded=386 net_sales=1936576.63
2026-08-18 14:56:32,540 | INFO | EXTRACT orders_batch_3 | rows=424 | started=2026-08-18T14:56:32 | ended=2026-08-18T14:56:32
2026-08-18 14:56:32,681 | INFO | BATCH 3 | read=424 valid=389 rejected=35 duplicated=3 loaded=385 net_sales=2841792.10



=== PIPELINE RUN LOG ===
 run_id  batch          started_at            ended_at  rows_read  rows_valid  rows_rejected  rows_duplicated  rows_loaded  total_net_sales  status
      1      1 2026-08-18T14:56:30 2026-08-18T14:56:31        420         389             31                0          389      982277.6805 SUCCESS
      2      1 2026-08-18T14:56:31 2026-08-18T14:56:31        420         389             31                0            0      982277.6805 SUCCESS
      3      2 2026-08-18T14:56:31 2026-08-18T14:56:32        424         387             37                1          386     1936576.6300 SUCCESS
      4      3 2026-08-18T14:56:32 2026-08-18T14:56:32        424         389             35                3          385     2841792.0980 SUCCESS

=== FINAL KPI ===
 fact_rows  rejected_rows  total_net_sales  invalid_fact_rows
      1159            134      2841792.098                  0
